In [1]:
import polars as pl
import polars.selectors as cs

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
df_path = r'/mnt/samsung/Datasets/Student_Placement_Skills_2025/Student_Placement_Skills_2025.csv'

In [3]:
df = pl.read_csv(df_path)

In [4]:
df = pl.read_csv(
    df_path,
    schema_overrides={
        "Student_ID": pl.UInt32,
        "Technical_Skills_Score_100": pl.UInt8,
        "Communication_Skills_Score_100": pl.UInt8,
        "Aptitude_Test_Score_100": pl.UInt8,
        "Internships_Count":    pl.UInt8,
        "Projects_Count":       pl.UInt8,
        "Certifications_Count": pl.UInt8,
        "Salary_Offered_USD": pl.Float32,
        "Gender": pl.Categorical,
        "Degree": pl.Categorical,
        "Placement_Offer": pl.Categorical,
    }
)

In [5]:
df.estimated_size('mb')

0.0240325927734375

In [6]:
df = pl.scan_csv(
    df_path,
    schema_overrides={
        "Student_ID": pl.UInt32,
        "Technical_Skills_Score_100": pl.UInt8,
        "Communication_Skills_Score_100": pl.UInt8,
        "Aptitude_Test_Score_100": pl.UInt8,
        "Internships_Count":    pl.UInt8,
        "Projects_Count":       pl.UInt8,
        "Certifications_Count": pl.UInt8,
        "Salary_Offered_USD": pl.Float32,
        "Gender": pl.Categorical,
        "Degree": pl.Categorical,
        "Placement_Offer": pl.Categorical,
    }
)

In [7]:
print(df.explain())

Csv SCAN [/mnt/samsung/Datasets/Student_Placement_Skills_2025/Student_Placement_Skills_2025.csv]
PROJECT */13 COLUMNS
ESTIMATED ROWS: 663


In [8]:
# Build the lazy query — Polars will push these filters down to the CSV scan
lf = (
    pl.scan_csv(df_path)
    .filter(
        (pl.col("Degree") == "Engineering") &
        (pl.col("Placement_Offer") == "Yes") &
        (pl.col("CGPA") > 3.5)
    )
)

# Confirm predicate pushdown — look for SELECTION inside the CSV_SCAN node
print(lf.explain())


Csv SCAN [/mnt/samsung/Datasets/Student_Placement_Skills_2025/Student_Placement_Skills_2025.csv]
PROJECT */13 COLUMNS
SELECTION: [([([(col("Placement_Offer")) == ("Yes")]) & ([(col("Degree")) == ("Engineering")])]) & ([(col("CGPA")) > (3.5)])]
ESTIMATED ROWS: 663


In [9]:
# Collect and inspect the result
result = lf.collect()
print(f"Rows: {len(result)}")
result


Rows: 22


Student_ID,Gender,Age,Degree,CGPA,Internships_Count,Projects_Count,Certifications_Count,Technical_Skills_Score_100,Communication_Skills_Score_100,Aptitude_Test_Score_100,Placement_Offer,Salary_Offered_USD
i64,str,i64,str,f64,i64,i64,i64,i64,i64,i64,str,f64
2,"""Female""",27,"""Engineering""",3.66,0,5,2,78,54,40,"""Yes""",3518.56
111,"""Male""",27,"""Engineering""",3.74,4,6,4,75,99,80,"""Yes""",6350.75
115,"""Female""",22,"""Engineering""",3.97,0,8,0,41,60,88,"""Yes""",3936.27
198,"""Male""",24,"""Engineering""",3.89,2,2,2,83,97,63,"""Yes""",13654.84
199,"""Female""",22,"""Engineering""",3.98,4,1,1,70,49,65,"""Yes""",10037.77
…,…,…,…,…,…,…,…,…,…,…,…,…
493,"""Female""",20,"""Engineering""",3.76,1,5,3,41,66,81,"""Yes""",9258.06
508,"""Female""",23,"""Engineering""",3.98,0,6,5,85,71,52,"""Yes""",16395.53
536,"""Female""",20,"""Engineering""",3.94,2,8,0,95,96,74,"""Yes""",16141.42


In [10]:
df.collect()

Student_ID,Gender,Age,Degree,CGPA,Internships_Count,Projects_Count,Certifications_Count,Technical_Skills_Score_100,Communication_Skills_Score_100,Aptitude_Test_Score_100,Placement_Offer,Salary_Offered_USD
u32,cat,i64,cat,f64,u8,u8,u8,u8,u8,u8,cat,f32
1,"""Male""",19,"""Business""",2.56,3,8,0,64,42,57,"""Yes""",8047.080078
2,"""Female""",27,"""Engineering""",3.66,0,5,2,78,54,40,"""Yes""",3518.560059
3,"""Male""",26,"""Data Science""",3.73,0,5,1,61,54,49,"""No""",11791.75
4,"""Male""",18,"""Computer Science""",2.21,2,8,5,66,42,72,"""Yes""",13946.280273
5,"""Male""",20,"""Business""",2.59,3,9,2,69,50,53,"""No""",10951.660156
…,…,…,…,…,…,…,…,…,…,…,…,…
596,"""Female""",22,"""Business""",2.52,3,9,4,59,45,78,"""No""",14940.129883
597,"""Male""",19,"""Data Science""",2.86,3,9,1,73,81,63,"""Yes""",8607.620117
598,"""Male""",28,"""Data Science""",2.71,0,5,0,77,96,82,"""No""",16772.720703


In [11]:
df.filter(
    pl.col('Internships_Count') == 0
)

In [15]:
median_salary = df.select(pl.col('Salary_Offered_USD').median()).collect().item()
median_salary

11763.734375

In [16]:
print(f"Median Salary: ${median_salary:.2f}")

Median Salary: $11763.73


In [25]:
result = (
    df.filter(
        (pl.col('Internships_Count') == 0) &
        (pl.col('Certifications_Count') == 0) &
        (pl.col('Placement_Offer') == 'Yes') &
        (pl.col('Salary_Offered_USD') > median_salary)
    )
).collect(engine='streaming')

result

Student_ID,Gender,Age,Degree,CGPA,Internships_Count,Projects_Count,Certifications_Count,Technical_Skills_Score_100,Communication_Skills_Score_100,Aptitude_Test_Score_100,Placement_Offer,Salary_Offered_USD
u32,cat,i64,cat,f64,u8,u8,u8,u8,u8,u8,cat,f32
244,"""Female""",18,"""Engineering""",3.55,0,9,0,87,65,74,"""Yes""",13687.320312
451,"""Female""",22,"""Data Science""",3.68,0,6,0,63,45,56,"""Yes""",16743.460938
477,"""Male""",20,"""Arts""",2.31,0,5,0,66,46,46,"""Yes""",17161.060547
483,"""Male""",23,"""Arts""",2.5,0,1,0,98,69,63,"""Yes""",12323.299805
531,"""Female""",19,"""Arts""",3.83,0,9,0,86,80,50,"""Yes""",15166.75
